<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Backend Module 3: Auth — Hashing, JWT and Roles

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. See why passwords are **hashed, never encrypted** — and what a salt actually does
2. Build a **JWT**, read it without any secret, and then **fail to forge one**
3. Write `/register` and `/login`, and get a real token back
4. Protect an endpoint so it returns **401** without a token and **200** with one
5. Add **roles**, and see why the answer is **403**, not 401
6. Meet the security mistakes that get made for real, and the one-line fixes

> **Nothing to install beyond one pip line, and no server.** Same `TestClient` trick as the CRUD
> notebook — your app runs right here.

## 1. How to Use This Notebook

Run every cell in order. Sections 2 and 3 are the two ideas; everything after that is them being used.

**Change things.** Use the wrong password. Edit a token. Ask for a protected route without logging in.
Every failure in this notebook is one you want to have seen once, safely.

---

In [ ]:
!pip install -q fastapi "pydantic[email]" httpx sqlalchemy "pwdlib[bcrypt]" pyjwt
print("ready")

## 2. Hashing Is a One-Way Door

**Encryption** is a two-way door: what goes in comes back out, if you hold the key.
**Hashing** is one-way. You never get the password back — you only ever check whether a new attempt
produces the same result.

In [ ]:
from pwdlib import PasswordHash
from pwdlib.hashers.bcrypt import BcryptHasher

# pwdlib is what FastAPI's own docs use now. (The older passlib is unmaintained.)
password_hash = PasswordHash((BcryptHasher(),))

hash_1 = password_hash.hash("lpu-2026")
hash_2 = password_hash.hash("lpu-2026")        # the SAME password, hashed again

print("hash #1:", hash_1)
print("hash #2:", hash_2)
print()
print("Same password, different hashes:", hash_1 != hash_2)

💡 **Each hash carries its own random salt.** That is why two students who both picked `123456` do not
look identical in the database — and why one stolen table cannot be cracked all at once.

Checking a password does not decrypt anything. It re-hashes the attempt and compares.

In [ ]:
print("correct password:", password_hash.verify("lpu-2026", hash_1))
print("wrong password  :", password_hash.verify("lpu-2027", hash_1))
print()
print("Notice what does not exist: there is no unhash().")
print("If a website ever emails you your old password, they stored it in plain text.")

## 3. What a JWT Actually Is

Three chunks joined by dots: **`header.payload.signature`**.

In [ ]:
from datetime import datetime, timedelta, timezone
import jwt                                   # the PyJWT package

SECRET = "change-me-in-production"
ALGORITHM = "HS256"

token = jwt.encode(
    {"sub": "ada@lpu.in", "role": "admin",
     "exp": datetime.now(timezone.utc) + timedelta(minutes=30)},
    SECRET, algorithm=ALGORITHM)

print(token)
print()
print("three parts:", len(token.split(".")))

Now decode the middle chunk **with no secret at all**.

In [ ]:
import base64, json

payload_chunk = token.split(".")[1]
decoded = json.loads(base64.urlsafe_b64decode(payload_chunk + "=" * (-len(payload_chunk) % 4)))
print(decoded)
print()
print("A JWT is SIGNED, not ENCRYPTED. Anyone holding it can read it.")
print("So never put anything private in a token - no passwords, no card numbers.")

So if anyone can read it, what stops them editing it? Try.

In [ ]:
# An attacker rewrites the payload to make themselves an admin, keeping the original signature.
fake_payload = base64.urlsafe_b64encode(
    json.dumps({"sub": "attacker@lpu.in", "role": "admin"}).encode()).decode().rstrip("=")
header, _, signature = token.split(".")
forged = f"{header}.{fake_payload}.{signature}"

try:
    jwt.decode(forged, SECRET, algorithms=[ALGORITHM])
    print("it worked?! (it should not)")
except jwt.InvalidTokenError as e:
    print("rejected:", type(e).__name__)
    print()
    print("Readable, but not forgeable. That is the whole idea.")

## 4. A Real Login

Now put those two ideas into an app: a `users` table, `/register`, and `/login`.

In [ ]:
import os
if os.path.exists("auth.db"):
    os.remove("auth.db")

from sqlalchemy import Integer, String, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, sessionmaker

engine = create_engine("sqlite:///auth.db")
SessionLocal = sessionmaker(bind=engine)


class Base(DeclarativeBase):
    pass


class User(Base):
    __tablename__ = "users"
    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    email: Mapped[str] = mapped_column(String(120), unique=True)
    hashed_password: Mapped[str] = mapped_column(String(200))   # NOT "password". The name matters.
    role: Mapped[str] = mapped_column(String(20), default="student")


Base.metadata.create_all(bind=engine)


def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()


print("users table ready")

In [ ]:
from fastapi import Depends, FastAPI, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from fastapi.testclient import TestClient
from pydantic import BaseModel, ConfigDict, EmailStr, Field

app = FastAPI()
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="login")   # this is what draws the padlock in /docs


class RegisterIn(BaseModel):
    email: EmailStr
    password: str = Field(min_length=8)
    role: str = "student"


class UserOut(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int
    email: EmailStr
    role: str          # no password field anywhere. On purpose.


@app.post("/register", status_code=201, response_model=UserOut)
def register(incoming: RegisterIn, db: Session = Depends(get_db)):
    if db.scalar(select(User).where(User.email == incoming.email)):
        raise HTTPException(409, "That email is already registered")
    user = User(email=incoming.email,
                hashed_password=password_hash.hash(incoming.password),   # hashed here, once
                role=incoming.role)
    db.add(user); db.commit(); db.refresh(user)
    return user


client = TestClient(app)
print(client.post("/register", json={
    "email": "ada@lpu.in", "password": "lpu-2026-ok", "role": "admin"}).json())

Notice the response has **no password field of any kind** — not even the hash.

Check what actually went into the table:

In [ ]:
with SessionLocal() as db:
    row = db.scalar(select(User).where(User.email == "ada@lpu.in"))
    print("stored as:", row.hashed_password)
    print()
    print("The plain password 'lpu-2026-ok' is nowhere in the database, and cannot be recovered.")

## 5. `/login` — Password In, Token Out

In [ ]:
def create_access_token(email: str, role: str) -> str:
    return jwt.encode({"sub": email, "role": role,
                       "exp": datetime.now(timezone.utc) + timedelta(minutes=30)},
                      SECRET, algorithm=ALGORITHM)


@app.post("/login")
def login(form: OAuth2PasswordRequestForm = Depends(), db: Session = Depends(get_db)):
    user = db.scalar(select(User).where(User.email == form.username))
    # ONE message for both failures. "No such user" would tell an attacker
    # which emails are registered - that is an account-enumeration leak.
    if user is None or not password_hash.verify(form.password, user.hashed_password):
        raise HTTPException(401, "Incorrect email or password")
    return {"access_token": create_access_token(user.email, user.role), "token_type": "bearer"}


# NOTE: data=, not json= - OAuth2 sends form fields. This trips everyone up once.
r = client.post("/login", data={"username": "ada@lpu.in", "password": "lpu-2026-ok"})
my_token = r.json()["access_token"]
print("status:", r.status_code)
print("token :", my_token[:45], "...")

In [ ]:
print("wrong password:", client.post("/login", data={
    "username": "ada@lpu.in", "password": "not-the-one"}).status_code)
print("unknown user  :", client.post("/login", data={
    "username": "ghost@lpu.in", "password": "lpu-2026-ok"}).status_code)
print()
print("Same code, and the same message - deliberately.")

## 6. Protecting an Endpoint

`get_current_user` turns a token back into a user, or refuses. Look at its shape: it is `get_db`
with a different body. **Nothing new was invented for auth.**

In [ ]:
def get_current_user(token: str = Depends(oauth2_scheme),
                     db: Session = Depends(get_db)) -> User:
    bad = HTTPException(401, "Could not validate credentials")
    try:
        payload = jwt.decode(token, SECRET, algorithms=[ALGORITHM])
    except jwt.InvalidTokenError:
        raise bad                    # tampered, expired or wrong secret - all the same answer
    user = db.scalar(select(User).where(User.email == payload.get("sub")))
    if user is None:
        raise bad
    return user


@app.get("/me", response_model=UserOut)
def me(user: User = Depends(get_current_user)):
    return user                      # no token-checking code in here at all


print("no token   :", client.get("/me").status_code)
print("bad token  :", client.get("/me", headers={"Authorization": "Bearer nonsense"}).status_code)
print("real token :", client.get("/me", headers={"Authorization": f"Bearer {my_token}"}).status_code)
print()
print(client.get("/me", headers={"Authorization": f"Bearer {my_token}"}).json())

## 7. Roles — 403 Is Not 401

- **401 Unauthorized** — *"I don't know who you are."* (Badly named. It means unauthenticated.)
- **403 Forbidden** — *"I know exactly who you are, and no."*

In [ ]:
def require_role(*allowed: str):
    def checker(user: User = Depends(get_current_user)) -> User:
        if user.role not in allowed:
            raise HTTPException(403, f"This needs one of: {', '.join(allowed)}")
        return user
    return checker


@app.delete("/courses/{code}", status_code=204,
            dependencies=[Depends(require_role("admin"))])
def delete_course(code: str):
    return None


client.post("/register", json={"email": "raj@lpu.in", "password": "lpu-2026-ok", "role": "student"})
student_token = client.post("/login", data={
    "username": "raj@lpu.in", "password": "lpu-2026-ok"}).json()["access_token"]

print("admin   deletes:", client.delete(
    "/courses/CSE101", headers={"Authorization": f"Bearer {my_token}"}).status_code)
print("student deletes:", client.delete(
    "/courses/CSE101", headers={"Authorization": f"Bearer {student_token}"}).status_code)
print("nobody  deletes:", client.delete("/courses/CSE101").status_code)

**204 · 403 · 401** — three different answers to three genuinely different situations.

## 8. The Rules That Are All Real Mistakes

| Rule | Why |
|---|---|
| Secrets from the environment, never the code | A key in git is a key on the internet. Bots scan public repos within minutes. |
| Never log a token or a password | Logs get shipped to third parties and pasted into tickets. |
| Never return a raw exception | Stack traces leak table names, paths and versions. |
| HTTPS only | A bearer token over plain HTTP is a password shouted across a cafe. |
| Real CORS origins in production | `*` is fine while learning and wrong when live. |
| Rate-limit `/login` | Otherwise it is an unlimited guessing machine. |
| Same error for bad user and bad password | Or you have published your user list. |
| Short token lifetime | A leaked token then expires by itself. |

> The `SECRET` in this notebook is in the code, which breaks rule 1. That is fine here and **not** fine
> anywhere real — on a deployed app it comes from an environment variable.

---

## 9. Exercises

### Q1. Register yourself as a `student`

**Hint:** the password has a minimum length of 8. What comes back if it is shorter?

In [ ]:
r = client.post("/register", json={
    "email": "___@lpu.in", "password": "___", "role": "___"})
print(r.status_code, r.json())

### Q2. Log in as yourself and keep the token

**Hint:** `data=`, not `json=`. OAuth2 uses form fields.

In [ ]:
r = client.post("/login", ___={"username": "___@lpu.in", "password": "___"})
my_new_token = r.json()["___"]
print(my_new_token[:40], "...")

### Q3. Call `/me` with your token and print your role

**Hint:** the header is `Authorization: Bearer <token>`.

In [ ]:
r = client.get("/me", headers={"___": f"Bearer {my_new_token}"})
print(r.json()["___"])

### Q4. Try to delete a course as a student, and predict the number first

**Hint:** you are authenticated perfectly well. You are just not allowed.

In [ ]:
print(client.delete("/courses/ECE201",
                    headers={"Authorization": f"Bearer {my_new_token}"}).status_code)

---

## Key Takeaways

1. **Hashing is one-way.** There is no `unhash()`, and every hash carries its own salt.
2. **A JWT is signed, not encrypted.** Anyone can read it; nobody can change it. So never put secrets inside one.
3. **`get_current_user` is `get_db` with a different body** — auth is a variation of dependency injection, not a new subject.
4. **401 means "who are you?", 403 means "not you".** Different questions, different answers.
5. **One error message for both login failures**, or you have leaked your user list.
6. **The secret belongs in the environment**, never in the code you push.

> Next: [DEPLOY.md](https://github.com/Poorit-Technologies/lpu-training/blob/main/backend-engineering/DEPLOY.md) — put this on the internet and open it on your phone.